## Pipeline Configuration

In [0]:
# ETL Pipeline Configuration

from datetime import datetime

PROJECT_NAME = "Olist E-Commerce Data Engineering"
DATABASE = "workspace.ecommerce_de_project"

PIPELINE_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PIPELINE_START_TIME = datetime.now()

print("Pipeline:", PROJECT_NAME)
print("Run ID:", PIPELINE_RUN_ID)
print("Start Time:", PIPELINE_START_TIME)
print("Database:", DATABASE)

Pipeline: Olist E-Commerce Data Engineering
Run ID: 20261125_045616
Start Time: 2026-11-25 04:56:16.341518
Database: workspace.ecommerce_de_project


## Pipeline Logging

In [0]:
# Pipeline Logging

pipeline_status = "RUNNING"

print("================================")
print("PIPELINE EXECUTION LOG")
print("================================")
print("Pipeline Name:", PROJECT_NAME)
print("Run ID:", PIPELINE_RUN_ID)
print("Status:", pipeline_status)
print("Started At:", PIPELINE_START_TIME)
print("================================")

PIPELINE EXECUTION LOG
Pipeline Name: Olist E-Commerce Data Engineering
Run ID: 20261125_045616
Status: RUNNING
Started At: 2026-11-25 04:56:16.341518


## Pipeline-এর প্রথম বাস্তব Task: Silver Data Validation

In [0]:
# Task 1 — Silver Data Validation

orders_table = f"{DATABASE}.silver_orders"
order_items_table = f"{DATABASE}.silver_order_items"

orders_count = spark.table(orders_table).count()
order_items_count = spark.table(order_items_table).count()

print("SILVER DATA VALIDATION")
print("======================")
print("Silver Orders:", orders_count)
print("Silver Order Items:", order_items_count)

if orders_count > 0 and order_items_count > 0:
    print("VALIDATION STATUS: PASSED")
else:
    print("VALIDATION STATUS: FAILED")

SILVER DATA VALIDATION
Silver Orders: 99433
Silver Order Items: 112650
VALIDATION STATUS: PASSED


In [0]:
# Task 2 — Critical Data Integrity Check

orders_null_ids = spark.sql(f"""
    SELECT COUNT(*) AS null_count
    FROM {orders_table}
    WHERE order_id IS NULL
       OR customer_id IS NULL
""").collect()[0]["null_count"]

items_null_ids = spark.sql(f"""
    SELECT COUNT(*) AS null_count
    FROM {order_items_table}
    WHERE order_id IS NULL
       OR product_id IS NULL
       OR seller_id IS NULL
""").collect()[0]["null_count"]

print("DATA INTEGRITY CHECK")
print("===================")
print("Orders null IDs:", orders_null_ids)
print("Order Items null IDs:", items_null_ids)

if orders_null_ids == 0 and items_null_ids == 0:
    print("INTEGRITY STATUS: PASSED")
else:
    print("INTEGRITY STATUS: FAILED")

DATA INTEGRITY CHECK
Orders null IDs: 0
Order Items null IDs: 0
INTEGRITY STATUS: PASSED


## Pipeline Quality Gate

In [0]:
# Pipeline Quality Gate

quality_checks = {
    "Silver Orders Available": orders_count > 0,
    "Silver Order Items Available": order_items_count > 0,
    "Orders ID Integrity": orders_null_ids == 0,
    "Order Items ID Integrity": items_null_ids == 0
}

print("PIPELINE QUALITY GATE")
print("====================")

for check_name, result in quality_checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{check_name}: {status}")

if all(quality_checks.values()):
    print("\nQUALITY GATE: PASSED")
else:
    raise Exception("PIPELINE FAILED: Quality checks did not pass")

PIPELINE QUALITY GATE
Silver Orders Available: PASS
Silver Order Items Available: PASS
Orders ID Integrity: PASS
Order Items ID Integrity: PASS

QUALITY GATE: PASSED


## Gold Layer Validation

In [0]:
# Gold Layer Validation

gold_tables = [
    "gold_monthly_orders",
    "gold_monthly_revenue",
    "gold_monthly_freight",
    "gold_order_status",
    "gold_seller_performance",
    "gold_top_products"
]

gold_validation = {}

for table in gold_tables:
    full_table_name = f"{DATABASE}.{table}"
    exists = spark.catalog.tableExists(full_table_name)
    gold_validation[table] = exists

    status = "PASS" if exists else "FAIL"
    print(f"{table}: {status}")

if all(gold_validation.values()):
    print("\nGOLD LAYER VALIDATION: PASSED")
else:
    print("\nGOLD LAYER VALIDATION: FAILED")

gold_monthly_orders: PASS
gold_monthly_revenue: PASS
gold_monthly_freight: PASS
gold_order_status: PASS
gold_seller_performance: PASS
gold_top_products: PASS

GOLD LAYER VALIDATION: PASSED


## Gold Data Integrity Check

In [0]:
# Gold Data Integrity Check

gold_row_counts = {}

for table in gold_tables:
    full_table_name = f"{DATABASE}.{table}"
    row_count = spark.table(full_table_name).count()
    gold_row_counts[table] = row_count

    status = "PASS" if row_count > 0 else "FAIL"
    print(f"{table}: {row_count} rows | {status}")

if all(count > 0 for count in gold_row_counts.values()):
    print("\nGOLD DATA INTEGRITY: PASSED")
else:
    raise Exception("GOLD DATA INTEGRITY FAILED")

gold_monthly_orders: 25 rows | PASS
gold_monthly_revenue: 24 rows | PASS
gold_monthly_freight: 24 rows | PASS
gold_order_status: 8 rows | PASS
gold_seller_performance: 3095 rows | PASS
gold_top_products: 32951 rows | PASS

GOLD DATA INTEGRITY: PASSED


## Pipeline Final Execution Summary

In [0]:
# Pipeline Final Execution Summary

PIPELINE_END_TIME = datetime.now()
PIPELINE_DURATION = (PIPELINE_END_TIME - PIPELINE_START_TIME).total_seconds()

silver_passed = all(quality_checks.values())
gold_passed = all(gold_validation.values()) and all(
    count > 0 for count in gold_row_counts.values()
)

if silver_passed and gold_passed:
    pipeline_status = "SUCCESS"
else:
    pipeline_status = "FAILED"

print("=" * 45)
print("FINAL PIPELINE EXECUTION SUMMARY")
print("=" * 45)
print("Pipeline:", PROJECT_NAME)
print("Run ID:", PIPELINE_RUN_ID)
print("Status:", pipeline_status)
print("Start Time:", PIPELINE_START_TIME)
print("End Time:", PIPELINE_END_TIME)
print("Duration:", round(PIPELINE_DURATION, 2), "seconds")
print("=" * 45)

FINAL PIPELINE EXECUTION SUMMARY
Pipeline: Olist E-Commerce Data Engineering
Run ID: 20261125_045616
Status: SUCCESS
Start Time: 2026-11-25 04:56:16.341518
End Time: 2026-11-25 05:12:11.827707
Duration: 955.49 seconds


## Persistent Pipeline Logging

In [0]:
# Persistent Pipeline Run Logging

log_table = f"{DATABASE}.pipeline_run_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {log_table} (
    pipeline_name STRING,
    run_id STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds DOUBLE,
    logged_at TIMESTAMP
)
USING DELTA
""")

log_data = [(
    PROJECT_NAME,
    PIPELINE_RUN_ID,
    pipeline_status,
    PIPELINE_START_TIME,
    PIPELINE_END_TIME,
    float(PIPELINE_DURATION),
    datetime.now()
)]

log_columns = [
    "pipeline_name",
    "run_id",
    "status",
    "start_time",
    "end_time",
    "duration_seconds",
    "logged_at"
]

log_df = spark.createDataFrame(log_data, log_columns)
log_df.createOrReplaceTempView("current_pipeline_log")

spark.sql(f"""
MERGE INTO {log_table} AS target
USING current_pipeline_log AS source
ON target.run_id = source.run_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("Pipeline run logged successfully!")
print("Run ID:", PIPELINE_RUN_ID)

Pipeline run logged successfully!
Run ID: 20261125_045616


## Verify Pipeline Run Log

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.pipeline_run_log
ORDER BY logged_at DESC;

pipeline_name,run_id,status,start_time,end_time,duration_seconds,logged_at
Olist E-Commerce Data Engineering,20261125_045616,SUCCESS,2026-11-25T04:56:16.341Z,2026-11-25T05:12:11.827Z,955.486189,2026-11-25T05:13:42.152Z
